# SA-CUT — E1 Colab training (Ch5 main experiment)

Trains the **full SA-CUT model** (E1 in the Ch5 experiment matrix): the reference run
that all baselines (E2) and ablations (E3) are compared against.

**Run order: Cell 1 → 2 → 3 → 4 (smoke) → 5A (short, 20 epochs) → 5B (full, 400 epochs).**
5A is not optional boilerplate — it is the checkpoint where you confirm `loss_D` is healthy
before committing hours of A100 time.

**Why this notebook instead of `SA_CUT_Colab_Train.ipynb`:** that notebook builds its
experiment YAML from a Python dict listing only four loss keys, so anything not in that
list silently falls back to `configs/default.yaml`. Since `default.yaml` keeps
`color_loss_mode: global` (so ablation baselines stay unchanged), that path would train
**without the region-conditioned colour loss**. This notebook uses
`configs/experiment_sa_cut_full.yaml` directly and only overrides paths on the CLI, so
every component stays exactly as committed.

**Design (same three rules as the SQ-MIL bootstrap):**
1. **Code on Colab local disk**, pulled from GitHub — fast, always the committed config.
2. **Data copied Drive → local disk once per session** — training reads many small patch
   files per epoch; the Drive FUSE mount is far slower than local disk.
3. **Checkpoints written back to Drive** — Colab disconnects; weights must survive.

**Watch while training (the mandatory training rule):** `loss_D` must stay in
**0.3–0.7**. If it falls below 0.1 the adversarial gradient to G vanishes and G degrades
to "colourised TPAF" instead of H&E. `d_loss_gate_threshold=0.1` auto-skips D updates to
recover, but check the `D=` and `D_gate=` fields in the epoch log.

Expected on a healthy run: `struct=0.0000` for the first 3 epochs (warm-up) then ramping
to `lambda_struct=5.0` over 5 epochs; `color` noticeably larger than in `global` mode.

In [ ]:
# ── Cell 1: mount Drive + config ─────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

# --- edit here if your paths differ ---
os.environ['DRIVE']    = '/content/drive/MyDrive/SA-CUT'
os.environ['REPO_URL'] = 'https://github.com/z-pan/SA-CUT.git'
os.environ['BRANCH']   = 'main'
os.environ['RUN_NAME']  = 'E1_sa_cut_full'

assert os.path.isdir(os.environ['DRIVE']), f"Drive folder not found: {os.environ['DRIVE']}"
print('GPU:'); os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')

In [ ]:
# ── Cell 2: code → local disk (clone, or pull if present) + deps ──
%cd /content
!if [ -d SA-CUT ]; then cd SA-CUT && git fetch && git checkout "$BRANCH" && git pull --ff-only; else git clone --branch "$BRANCH" "$REPO_URL"; fi
%cd /content/SA-CUT
!pip install -q tifffile pytorch-fid pyyaml scikit-image
# Confirm the region-conditioned colour loss commit is present.
!git log --oneline -3
!test -f losses/region_color_loss.py && echo 'OK: region_color_loss.py present'

In [ ]:
# ── Cell 3: data Drive → local disk (once per session) ──
# rsync --ignore-existing makes reconnects near-instant.
!mkdir -p data/raw/tpaf data/raw/he data/patches/masks
!rsync -a --ignore-existing "$DRIVE/patches/tpaf/"  data/raw/tpaf/
!rsync -a --ignore-existing "$DRIVE/patches/he/"    data/raw/he/
!rsync -a --ignore-existing "$DRIVE/patches/masks/" data/patches/masks/

# Masks are matched to TPAF patches by filename stem, so a mask must exist for
# every TPAF patch in precomputed mode.
!echo "tpaf=$(ls data/raw/tpaf | wc -l)  he=$(ls data/raw/he | wc -l)  masks=$(ls data/patches/masks | wc -l)"

In [ ]:
# ── Cell 4: smoke test first (~30 s) ──
# Same check as locally: CPU, 1 epoch, tiny synthetic data. Verifies the Colab
# environment and the committed code before spending A100 hours.
!bash scripts/smoke_test.sh --config configs/experiment_sa_cut_full.yaml

## STEP 5A — short validation run (RUN THIS FIRST)

**Do not skip ahead to the full run.** 20 epochs at constant LR, ~1/20th of the cost, to
confirm the run is healthy before committing hours of A100 time.

Check in the epoch log:

| Field | Healthy | Bad |
|---|---|---|
| `D=` | **0.3–0.7** | `< 0.1` → D collapse; G degrades to colourised TPAF |
| `D_gate=` | low % | pegged near 100% → D permanently gated |
| `struct=` | `0.0000` for epochs 0–2, then rising | still 0 after epoch 5 |
| `color=` | clearly non-trivial (region mode) | ~0 |
| `G=` | fluctuating, no NaN / blow-up | NaN or monotonic explosion |

Also open a sample image under `$DRIVE/results/logs/${RUN_NAME}_short/` — nuclei should be
trending purple, not blank/white (blank nuclei is the UTOM failure mode SA-CUT exists to fix).

Only if all of the above look right, continue to STEP 5B.

In [ ]:
# ── STEP 5A: short validation run — 20 epochs, no LR decay ──
# Separate RUN_NAME suffix so this never overwrites the real E1 checkpoints/logs.
!python scripts/train.py \
    --config configs/experiment_sa_cut_full.yaml \
    --training.n_epochs=20 \
    --training.n_epochs_decay=0 \
    --data.patch_size=512 \
    --data.num_workers=2 \
    --experiment.name="${RUN_NAME}_short" \
    --experiment.checkpoint_dir="$DRIVE/checkpoints/${RUN_NAME}_short" \
    --experiment.log_dir="$DRIVE/results/logs/${RUN_NAME}_short" \
    --experiment.use_wandb=false

## STEP 5B — full E1 run (only after 5A looks healthy)

400 epochs (200 at full LR + 200 linear decay) — the reference run for the Ch5 experiment
matrix. Expect at least one Colab disconnect; checkpoints go to Drive every 10 epochs, so
use the resume cell below.

The first 200 epochs run at constant LR, so **STEP 5A's epochs are numerically identical to
the opening of this run** (same seed, same LR). 5A therefore costs nothing in information
terms — it just lets you bail out early instead of hours in.

In [ ]:
# ── STEP 5B: full E1 training (400 epochs) ──
# Config is used as committed (mask input + SA-PatchNCE + L_struct + region colour
# loss + the D-collapse guards). Only paths and patch size are overridden.
# patch_size=512 matches the 512x512 precomputed masks on Drive.
!python scripts/train.py \
    --config configs/experiment_sa_cut_full.yaml \
    --data.patch_size=512 \
    --data.num_workers=2 \
    --experiment.name="$RUN_NAME" \
    --experiment.checkpoint_dir="$DRIVE/checkpoints/$RUN_NAME" \
    --experiment.log_dir="$DRIVE/results/logs/$RUN_NAME" \
    --experiment.use_wandb=false

## Resume after a disconnect

Re-run Cells 1–3 (Cell 3 is fast on reconnect), then run the cell below. Checkpoints live
on Drive, so a finished epoch is never lost.

In [ ]:
# ── STEP 6: resume the full E1 run from the last checkpoint ──
!python scripts/train.py \
    --config configs/experiment_sa_cut_full.yaml \
    --resume "$DRIVE/checkpoints/$RUN_NAME/latest.pth" \
    --data.patch_size=512 \
    --data.num_workers=2 \
    --experiment.name="$RUN_NAME" \
    --experiment.checkpoint_dir="$DRIVE/checkpoints/$RUN_NAME" \
    --experiment.log_dir="$DRIVE/results/logs/$RUN_NAME" \
    --experiment.use_wandb=false

## Next: E2 baselines and E3 ablations

Same pattern — swap the config, keep the path overrides and `RUN_NAME`. The ablation
configs already exist in `configs/`, so no dict-built YAML is needed:

| Experiment | Config |
|---|---|
| E2 CUT baseline | `configs/ablation_cut_baseline.yaml` |
| E2 CycleGAN | `configs/ablation_cyclegan.yaml` |
| E3 mask input only | `configs/ablation_cut_mask_input.yaml` |
| E3 no `L_struct` | `configs/ablation_sa_cut_no_struct.yaml` |

Give each run its own `RUN_NAME` so checkpoints and logs stay separate on Drive.